# 02 Business Rules Validation

This notebook validates business logic and service dependency rules:
- Partner/Dependents relationships
- PhoneService/MultipleLines consistency
- InternetService subscription dependency checks
- Cross-service validation

## Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../Data/raws/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

## 1. Partner and Dependents Relationship

In [ ]:
print(df['Partner'].unique()) 
print("\n")
print(df['Dependents'].unique())  

In [ ]:
# Count customers with dependents but no partner
dependents_no_partner = ((df['Dependents'] == 'Yes') & (df['Partner'] == 'No')).sum()
print(f"Customers with dependents but no partner: {dependents_no_partner}")
dependents_no_partner

In [ ]:
# Percentage within those who have dependents
pct = (
    (df['Dependents'] == 'Yes') &
    (df['Partner'] == 'No')
).sum() / (df['Dependents'] == 'Yes').sum()

print(f"Percentage of dependent-holders without partner: {pct:.1%}")
pct

**Insight:** ต้องระวัง! Partner กับ Dependent มีความไม่สัมพันทธ์กัน 361 คน อาจจะเป็นกลุ่มคนที่ (พ่อ/แม่เลี้ยงเดี่ยว, คนดูแลพ่อแม่, หย่าร้างแต่ยังมีลูก, อื่น ๆ)

## 2. PhoneService and MultipleLines Consistency

### Rule 1: Customers without PhoneService should have MultipleLines = "No phone service"

In [ ]:
print("people without phone service but with multiple lines:", 
      ((df["PhoneService"] == "No") & 
       (df["MultipleLines"] != "No phone service")).sum())

### Rule 2: MultipleLines = Yes requires PhoneService = Yes

In [ ]:
print("people with MultipleLines=Yes but PhoneService=No:",
      ((df["MultipleLines"] == "Yes") & 
       (df["PhoneService"] == "No")).sum())

## 3. Internet and Service Dependency Checks

### Rule 3: Customers must have at least one service (Phone OR Internet)

In [ ]:
print("amount of customers not have both InternetService and PhoneService :",
      ((df["InternetService"] == "No") &
       (df["PhoneService"] == "No")).sum())

**Finding:** จากการเช็คด้านบน ไม่มีใครที่ไม่มีทั้งเบอร์และอินเทอร์เน็ต แปลว่าทุกคนต้องมีอย่างน้อยหนึ่งอย่าง (มีเน็ต หรือ มีโทรศัพท์ อย่างใดอย่างหนึ่ง)

### Rule 4: OnlineSecurity availability when having Internet

In [ ]:
print("amount of customers not have OnlineSecurity but have InternetService :",
      ((df["OnlineSecurity"] == "No") &
       (df["InternetService"] != "No")).sum())

**Note:** มีลูกค้าจำนวนมากที่มี Internet แต่ไม่ได้เปิด OnlineSecurity (ซึ่งเป็นการเลือกที่ถูกต้องตาม business logic)

### Rule 5: OnlineBackup without OnlineSecurity

In [ ]:
print("amount of customers not have OnlineSecurity but have OnlineBackup :",
      ((df["OnlineSecurity"] == "No") &
       (df["OnlineBackup"] == "Yes")).sum())

**Finding:** มี customer ที่มี OnlineBackup แต่ไม่มี Online Security = 1303 คน

### Rule 6: DeviceProtection without OnlineSecurity

In [ ]:
print(
    "amount of customers have DeviceProtection without OnlineSecurity:",
    ((df["DeviceProtection"] == "Yes") & (df["OnlineSecurity"] == "No")).sum()
)

**Finding:** มี customer ที่มี DeviceProtection แต่ไม่มี Online Security = 1311 คน

## Summary of Business Rule Validation

✅ **All critical business rules passed:**
- No conflicts in PhoneService/MultipleLines
- Every customer has at least one service
- Service dependencies are logically consistent

📝 **Notable patterns:**
- 361 customers have dependents without a partner (17.1%)
- 1,303 customers choose OnlineBackup without OnlineSecurity
- 1,311 customers have DeviceProtection without OnlineSecurity

These patterns are valid and may represent:
- Single parents or guardians
- Customers prioritizing backup over security
- Device protection being more valuable than online security for certain segments

---

**Next:** [03_univariate_dist.ipynb](03_univariate_dist.ipynb) — Variable distributions and ranges